# EA-EMPD shocks: GC monetary events + ECB speeches → `empd_shock.csv`

Builds the pooled policy-communication shock series from the EA-EMPD
(Altavilla, Gürkaynak, Kind, Laeven 2025, `EA-EMPD.en (1).xlsx`).
Fully self-contained: inputs are the EA-EMPD file and `ois_2y.csv`
(ECB SDW export, used only as the trading-day calendar) — no dependence
on the release-shock pipeline.

**Construction decisions**
- Events kept: `GC_ME` (Monetary Event window — the paper's MP shock concept) plus
  `EB` and `P` speeches. `GC_PR`/`GC_PC` are dropped: they are sub-windows of ME and
  would double count the meeting.
- Shock = `OIS_2Y` window surprise (bp), same units as the paper's EA-MPD shock.
  Events with missing `OIS_2Y` are dropped (counted).
- Day assignment (panel `delta_y` is close-to-close): events at hour ≥ 18 CET or on
  non-trading days go to the **next** trading day; everything else (incl. pre-open
  morning events) to the same trading day. Borderline: 17h speeches stay same-day.
- Multiple events per assigned day are **summed** (net daily policy news, incl.
  speech + meeting on the same day).
- One-off cross-check (done at construction): the 38 GC_ME days inside the panel
  window match the paper's 38 EA-MPD `ecb_day` events 1:1, no misses either way.
  The cell below prints the GC_ME day list so this remains inspectable without
  importing anything from the release pipeline.

In [1]:
import pandas as pd
import numpy as np

events = pd.read_excel("EA-EMPD.en (1).xlsx", sheet_name="EA-EMPD")
events["Date_time"] = pd.to_datetime(events["Date_time"])

# trading-day calendar from the SDW OIS export (reverse-chronological, 5 header rows)
cal = pd.read_csv("ois_2y.csv", skiprows=5, header=None, usecols=[0], names=["date"])
cal["date"] = pd.to_datetime(cal["date"], errors="coerce")
cal = cal.dropna().sort_values("date").reset_index(drop=True)
caldates = cal["date"].values
print(f"calendar: {len(cal)} trading days, {cal['date'].min().date()} → {cal['date'].max().date()}")

calendar: 1430 trading days, 2021-01-04 → 2026-07-27


In [2]:
ev = events[
    events["Event_type"].isin(["GC_ME", "EB", "P"])
    & (events["Date_time"] >= "2020-12-15")
].copy()
n_all = len(ev)
n_gc = int((ev["Event_type"] == "GC_ME").sum())
ev = ev.dropna(subset=["OIS_2Y"])
print(f"events in window: {n_all} ({n_gc} GC_ME), dropped for missing OIS_2Y: {n_all - len(ev)}")

# target calendar date: next day if evening or non-trading day
shift = (ev["Date_time"].dt.hour >= 18) | (ev["Non_regular_trading_day"] == 1)
target = ev["Date_time"].dt.normalize() + pd.to_timedelta(shift.astype(int), unit="D")
print(f"events shifted to next day (evening/non-trading): {int(shift.sum())}")

# snap to first trading day >= target
idx = np.searchsorted(caldates, target.values, side="left")
in_cal = idx < len(caldates)
print(f"events beyond calendar end (dropped): {int((~in_cal).sum())}")
ev, idx = ev[in_cal].copy(), idx[in_cal]
ev["business_date"] = caldates[idx]

events in window: 847 (39 GC_ME), dropped for missing OIS_2Y: 61
events shifted to next day (evening/non-trading): 89
events beyond calendar end (dropped): 0


In [3]:
# GC days identified from the EA-EMPD itself (no external flag needed)
gc_days = pd.to_datetime(ev.loc[ev["Event_type"] == "GC_ME", "business_date"]).dt.date
print(f"GC_ME days: {len(gc_days)}")
print(sorted(gc_days))

GC_ME days: 39
[datetime.date(2021, 1, 21), datetime.date(2021, 3, 11), datetime.date(2021, 4, 22), datetime.date(2021, 6, 10), datetime.date(2021, 7, 22), datetime.date(2021, 9, 9), datetime.date(2021, 10, 28), datetime.date(2021, 12, 16), datetime.date(2022, 2, 3), datetime.date(2022, 3, 10), datetime.date(2022, 4, 14), datetime.date(2022, 6, 9), datetime.date(2022, 7, 21), datetime.date(2022, 9, 8), datetime.date(2022, 10, 27), datetime.date(2022, 12, 15), datetime.date(2023, 2, 2), datetime.date(2023, 3, 16), datetime.date(2023, 5, 4), datetime.date(2023, 6, 15), datetime.date(2023, 7, 27), datetime.date(2023, 9, 14), datetime.date(2023, 10, 26), datetime.date(2023, 12, 14), datetime.date(2024, 1, 25), datetime.date(2024, 3, 7), datetime.date(2024, 4, 11), datetime.date(2024, 6, 6), datetime.date(2024, 7, 18), datetime.date(2024, 9, 12), datetime.date(2024, 10, 17), datetime.date(2024, 12, 12), datetime.date(2025, 1, 30), datetime.date(2025, 3, 6), datetime.date(2025, 4, 17), datet

In [4]:
daily = (
    ev.groupby("business_date")
    .agg(empd_shock=("OIS_2Y", "sum"), n_empd_events=("OIS_2Y", "size"))
    .reset_index()
)

out = cal.rename(columns={"date": "business_date"}).merge(daily, on="business_date", how="left")
out["n_empd_events"] = out["n_empd_events"].fillna(0).astype(int)
out["empd_shock"] = out["empd_shock"].fillna(0.0)

evd = out[out["n_empd_events"] > 0]
print(f"trading days: {len(out)}, event days: {len(evd)}, multi-event days: {int((out['n_empd_events'] > 1).sum())}")
print("\nempd_shock on event days (bp):")
print(evd["empd_shock"].describe().round(3).to_string())
print("\n|shock| quantiles:", evd["empd_shock"].abs().quantile([0.5, 0.75, 0.9, 0.99]).round(2).tolist())

on_gc = evd["business_date"].dt.date.isin(set(gc_days))
print(f"\nby day type: GC days n={int(on_gc.sum())}, sd={evd.loc[on_gc, 'empd_shock'].std():.2f}bp | "
      f"speech-only days n={int((~on_gc).sum())}, sd={evd.loc[~on_gc, 'empd_shock'].std():.2f}bp")

trading days: 1430, event days: 534, multi-event days: 181

empd_shock on event days (bp):
count    534.000
mean      -0.174
std        2.652
min      -19.238
25%       -0.758
50%       -0.004
75%        0.599
max       17.958

|shock| quantiles: [0.67, 1.62, 3.13, 10.44]

by day type: GC days n=39, sd=6.61bp | speech-only days n=495, sd=2.05bp


In [5]:
out["business_date"] = out["business_date"].dt.strftime("%Y-%m-%d")
out[["business_date", "empd_shock", "n_empd_events"]].to_csv("empd_shock.csv", index=False)
print("written: empd_shock.csv")

written: empd_shock.csv
